# Week 4 EDA: RES vs NRES by metro

How imbalanced are residential vs non-residential signals across the 10 metros?

| | RES | NRES |
|---|---:|---:|
| Pure AE 10 m cells | 29,798 | 253,321 |
| Track-A buildings | 3,089 | 8,191 |
| AE-grid footprint area (km²) | 1.329 | 16.459 |
| Building footprint area (km²) | 2.181 | 27.240 |

Sources
- Pixels / AE-grid area: `datasets/week4_pixel_gt/pixels.parquet` (pure = one class only via `frac_*_full`)
- Buildings / building area: `datasets/building_bbox_grid/buildings.parquet` (`y_res`, `area_m2_approx`)

Rebuild:
```bash
.venv/bin/python -u scripts/build_week4_res_nres_eda.py
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "outputs" / "week4_res_nres_eda" / "manifest.json").exists():
        ROOT = p
        break

OUT = ROOT / "outputs" / "week4_res_nres_eda"
summary = pd.read_csv(OUT / "summary_by_region.csv")
manifest = json.loads((OUT / "manifest.json").read_text())
print("totals", manifest["totals"])
display(summary.round(3))


## Pure AE 10 m cells

After the unstratified GT rebuild, RES pure cells should dominate (natural mix). 1:1 balancing then undersamples RES to match NRES per metro.


![Pure pixels](../../outputs/week4_res_nres_eda/figures/pure_pixels_by_region.png)

![Pure share](../../outputs/week4_res_nres_eda/figures/pure_pixel_share_by_region.png)


## Buildings (Track-A labeled set)

Count of RES vs NRES buildings per metro, plus footprint area from `area_m2_approx`.

![Buildings](../../outputs/week4_res_nres_eda/figures/buildings_by_region.png)

![Building area](../../outputs/week4_res_nres_eda/figures/building_area_by_region.png)


## Footprint area on the AE grid

Sum of `area_res_m2` / `area_nres_m2` over Week-4 AE cells (polygon ∩ cell). This is the area that enters fraction GT.

![AE area](../../outputs/week4_res_nres_eda/figures/ae_footprint_area_by_region.png)

![Totals](../../outputs/week4_res_nres_eda/figures/totals_res_vs_nres.png)


## Metro table

| Metro | Pure RES cells | Pure NRES cells | RES bldgs | NRES bldgs | AE RES km² | AE NRES km² | Bldg RES km² | Bldg NRES km² |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Atlanta SE | 4,670 | 24,315 | 318 | 700 | 0.238 | 1.612 | 0.346 | 2.339 |
| Boston NE | 3,344 | 28,326 | 404 | 966 | 0.126 | 1.733 | 0.231 | 3.185 |
| Chicago MW | 3,068 | 37,057 | 349 | 972 | 0.144 | 2.640 | 0.261 | 4.776 |
| Denver W | 2,751 | 23,720 | 316 | 810 | 0.123 | 1.548 | 0.208 | 2.626 |
| Houston S | 4,132 | 34,127 | 325 | 846 | 0.203 | 2.294 | 0.270 | 3.054 |
| LA W | 1,544 | 18,315 | 185 | 842 | 0.063 | 1.065 | 0.092 | 1.556 |
| NYC NE | 2,767 | 29,342 | 337 | 990 | 0.105 | 1.838 | 0.183 | 3.213 |
| Phoenix SW | 2,863 | 22,236 | 308 | 772 | 0.125 | 1.478 | 0.181 | 2.131 |
| SF W | 1,411 | 16,445 | 203 | 602 | 0.056 | 1.024 | 0.089 | 1.643 |
| Seattle NW | 3,248 | 19,438 | 344 | 691 | 0.145 | 1.227 | 0.320 | 2.718 |
| **ALL** | **29,798** | **253,321** | **3,089** | **8,191** | **1.329** | **16.459** | **2.181** | **27.240** |


## Global / unstratified trend

Polygon GT is now unstratified Track-A (no per-class cap), so it matches the usual global pattern: residential is the majority. The older ~80/class stratified draw (~27% RES) is retired.

| Context | RES | NRES | % RES (count) | % RES (area) |
|---|---:|---:|---:|---:|
| Unstratified Overture, same AOIs (typed class) | 443,080 | 48,964 | **90%** | **59%** |
| US Census places (typed study labels) | 488,241 | 55,624 | **90%** | — |
| GHSL / UN DESA (built-up surface) | — | — | — | **~90%** |
| Legacy stratified Track-A (retired) | 3,089 | 8,191 | 27% | ~7% |

1:1 training balance undersamples RES pixels later — it does not invent an NRES-majority world.

![Global vs stratified](../../outputs/week4_res_nres_eda/figures/global_vs_stratified_building_mix.png)

![Unstratified counts](../../outputs/week4_res_nres_eda/figures/unstratified_buildings_by_region.png)

![Unstratified area](../../outputs/week4_res_nres_eda/figures/unstratified_building_area_by_region.png)

![US regions](../../outputs/week4_res_nres_eda/figures/us_region_typed_res_share.png)


## Takeaways

1. Unstratified Track-A GT keeps the natural RES majority (~90% by count in these AOIs).
2. Same AOIs / US typed places / GHSL–UN DESA all point the same way: residential dominates built-up.
3. Week-4 1:1 balancing undersamples RES pixels for fair F1 — it does not claim NRES is the global majority.
